In [1]:
import os
import pandas as pd

DATA_CANDIDATES = [
    "/kaggle/input/hotel-id-2021-fgvc8",
    "/kaggle/data/hotel-id-2021-fgvc8",
    "/kaggle/input",
    "/kaggle/data",
]


def find_file(filename: str):
    for base in DATA_CANDIDATES:
        path = os.path.join(base, filename)
        if os.path.exists(path):
            return path
    for base in DATA_CANDIDATES:
        path = os.path.join(base, "hotel-id-2021-fgvc8", filename)
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"Could not find {filename} under any of: {DATA_CANDIDATES}"
    )


train_path = find_file("train.csv")
sub_path = find_file("sample_submission.csv")

print("Using train:", train_path)
print("Using sample_submission:", sub_path)

train = pd.read_csv(train_path)
sample_sub = pd.read_csv(sub_path)

train.head(), sample_sub.head(), train.shape, sample_sub.shape



Using train: /kaggle/input/hotel-id-2021-fgvc8/train.csv
Using sample_submission: /kaggle/input/hotel-id-2021-fgvc8/sample_submission.csv


(                  image  chain  hotel_id            timestamp
 0  d29287f52c2a871f.jpg      5     22408  2018-04-16 17:01:49
 1  e9d067c249e4c2f9.jpg     70      2324  2016-07-08 22:26:21
 2  cc9877a40a63ed93.jpg      4     47514  2017-04-14 02:28:56
 3  a4963bc1d337d0c9.jpg      0     45612  2018-11-19 05:04:41
 4  c0ef87b07b40cdb2.jpg      6     16341  2017-09-27 19:43:51,
                   image                       hotel_id
 0  f1608c9f17fb6920.jpg  36363 53586 18807 64314 60181
 1  c6c63939c67931e1.jpg  36363 53586 18807 64314 60181
 2  83c214f3e90717ed.jpg  36363 53586 18807 64314 60181
 3  a6b5dbe436592c82.jpg  36363 53586 18807 64314 60181
 4  dd2f02f4354993cc.jpg  36363 53586 18807 64314 60181,
 (87798, 4),
 (9756, 2))

In [2]:
import os

global_top5 = train["hotel_id"].value_counts().head(5).index.astype(str).tolist()
global_pred_str = " ".join(global_top5)
print("Global top-5 fallback:", global_pred_str)

train_meta = train.loc[:, ["chain", "hotel_id"]].copy()
train_meta["chain"] = train_meta["chain"].astype(int)
train_meta["hotel_id"] = train_meta["hotel_id"].astype(int)

_chain_top5_df = (
    train_meta.groupby(["chain", "hotel_id"], sort=False)
    .size()
    .rename("cnt")
    .reset_index()
    .sort_values(
        ["chain", "cnt", "hotel_id"], ascending=[True, False, True], kind="mergesort"
    )
)
_chain_top5_df = (
    _chain_top5_df.groupby("chain", sort=False).head(5).reset_index(drop=True)
)

chain_to_top5_hotels = (
    _chain_top5_df.groupby("chain", sort=False)["hotel_id"]
    .apply(lambda s: [str(x) for x in s.tolist()])
    .to_dict()
)

print("Prepared chain->top5 map for", len(chain_to_top5_hotels), "chains (including 0)")

TRAIN_IMAGES_CANDIDATES = [
    "/kaggle/input/hotel-id-2021-fgvc8/train_images",
    "/kaggle/data/hotel-id-2021-fgvc8/train_images",
    "/kaggle/input/train_images",
    "/kaggle/data/train_images",
    "/kaggle/input/hotel-id-2021-fgvc8/train_images/train_images",
    "/kaggle/data/hotel-id-2021-fgvc8/train_images/train_images",
]
TEST_IMAGES_CANDIDATES = [
    "/kaggle/input/hotel-id-2021-fgvc8/test_images",
    "/kaggle/data/hotel-id-2021-fgvc8/test_images",
    "/kaggle/input/test_images",
    "/kaggle/data/test_images",
    "/kaggle/input/hotel-id-2021-fgvc8/test_images/test_images",
    "/kaggle/data/hotel-id-2021-fgvcvc8/test_images/test_images",
    "/kaggle/data/hotel-id-2021-fgvc8/test_images/test_images",
    "/kaggle/input/test_images/test_images",
    "/kaggle/data/test_images/test_images",
]

train_images_root = None
for p in TRAIN_IMAGES_CANDIDATES:
    if os.path.isdir(p):
        train_images_root = p
        break

test_images_root = None
for p in TEST_IMAGES_CANDIDATES:
    if os.path.isdir(p):
        test_images_root = p
        break

print("Using train_images root:", train_images_root)
print("Using test_images root:", test_images_root)

from PIL import Image
import numpy as np
import zlib


# --- SPEED: Make aHash computation allocation-light while preserving identical semantics.
# The original computed mean on float32 and compared arr > mean; we keep that exact logic
# but avoid extra reshapes/packbits/view overhead by building the uint64 bitmask directly.
def _ahash_64(image_path: str):
    """64-bit average hash (8x8) -> Python int; returns None if unreadable."""
    try:
        with Image.open(image_path) as im:
            im = im.convert("L").resize((8, 8), Image.BILINEAR)
            arr = np.asarray(im, dtype=np.float32).ravel()
        mean = float(arr.mean())
        bits = arr > mean  # bool[64], same thresholding as original
        h = 0
        # big-endian bit order: first element is the MSB
        for b in bits:
            h = (h << 1) | int(bool(b))
        return int(h)
    except Exception:
        return None


HASH_TRAIN_CAP = 90000
PER_CHAIN_CAP = 1200

_hash_to_chain_counts = {}
_hash_ready = False

_hash_keys_u64 = None
_hash_keys_py = None

_popcount_lut = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


# --- SPEED: Keep exact popcount but reuse a precomputed byte-view of the keys to reduce per-query overhead.
_hash_keys_bytes = None


def _popcount_u64_vectorized_xor_bytes(xor_u64: np.ndarray) -> np.ndarray:
    """Exact popcount for uint64 array via byte LUT."""
    xb = xor_u64.view(np.uint8).reshape(-1, 8)
    return _popcount_lut[xb].sum(axis=1).astype(np.int16)


from concurrent.futures import ThreadPoolExecutor

if train_images_root is not None and test_images_root is not None:
    try:
        train2 = train.loc[:, ["image", "chain"]].copy()
        train2["image"] = train2["image"].astype(str)
        train2["chain"] = train2["chain"].astype(int)

        def _stable_u32(s: str) -> int:
            return int(zlib.crc32(s.encode("utf-8")) & 0xFFFFFFFF)

        train2["__h"] = train2["image"].map(_stable_u32).astype(np.uint32)

        train2 = (
            train2.sort_values(["chain", "__h", "image"], kind="mergesort")
            .groupby("chain", sort=False)
            .head(PER_CHAIN_CAP)
            .reset_index(drop=True)
        )
        train2 = train2.drop(columns="__h")

        if len(train2) > HASH_TRAIN_CAP:
            train2 = train2.iloc[:HASH_TRAIN_CAP].copy()

        train2["chain_str"] = train2["chain"].astype(str)
        train2["path"] = (
            train_images_root.rstrip("/")
            + "/"
            + train2["chain_str"]
            + "/"
            + train2["image"]
        )

        # --- SPEED: Avoid an extra full pass of os.path.exists over up to 90k paths.
        # _ahash_64 already safely returns None on missing/unreadable files, preserving correctness.
        def _hash_one(path_and_chain):
            path, chain_val = path_and_chain
            h = _ahash_64(path)
            return h, int(chain_val)

        paths = train2["path"].tolist()
        chains = train2["chain"].tolist()

        max_workers = min(16, max(1, (os.cpu_count() or 4)))

        n_hashed = 0
        get_bucket = _hash_to_chain_counts.get
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            for h, ch in ex.map(_hash_one, zip(paths, chains), chunksize=256):
                if h is None:
                    continue
                d = get_bucket(h)
                if d is None:
                    d = {}
                    _hash_to_chain_counts[h] = d
                d[ch] = d.get(ch, 0) + 1
                n_hashed += 1

        _hash_ready = n_hashed > 0
        if _hash_ready:
            _hash_keys_py = list(_hash_to_chain_counts.keys())
            _hash_keys_u64 = np.array(_hash_keys_py, dtype=np.uint64)
            _hash_keys_bytes = _hash_keys_u64.view(np.uint8).reshape(-1, 8).copy()

        print(
            f"Built train hash index (hash->chain): hashed {n_hashed} train images (cap={HASH_TRAIN_CAP}, per_chain={PER_CHAIN_CAP})."
        )
    except Exception as e:
        print(
            "WARNING: hash index build failed; falling back to global-only predictions. Error:",
            repr(e),
        )
        _hash_ready = False
else:
    print(
        "WARNING: train_images/test_images roots not found; falling back to global-only predictions."
    )

_test_hash_cache = {}

TOPK_NEIGHBORS = 25
HAMMING_MAX_DIST = 14


def infer_chains_for_test_hash(hq: int):
    """Infer ranked chain candidates via hash NN voting from a precomputed test hash; returns list[int]."""
    if not _hash_ready:
        return []
    if hq is None:
        return []

    counts_exact = _hash_to_chain_counts.get(hq)
    if counts_exact is not None:
        ranked = sorted(counts_exact.items(), key=lambda kv: (-kv[1], kv[0]))
        return [ch for ch, _ in ranked]

    if _hash_keys_u64 is None or _hash_keys_u64.size == 0:
        return []

    q = np.uint64(hq)
    dists = _popcount_u64_vectorized_xor_bytes(_hash_keys_u64 ^ q)

    K = min(TOPK_NEIGHBORS, int(_hash_keys_u64.size))
    nn_idx = np.argpartition(dists, K - 1)[:K]
    nn_idx = nn_idx[np.argsort(dists[nn_idx], kind="mergesort")]

    chain_votes = {}
    get_counts = _hash_to_chain_counts.get
    ham_max = HAMMING_MAX_DIST
    for j in nn_idx:
        dj = int(dists[j])
        if dj > ham_max:
            break
        best_h = int(_hash_keys_u64[int(j)])
        counts = get_counts(best_h, {})
        w = (ham_max - dj) + 1
        for ch, c in counts.items():
            chain_votes[ch] = chain_votes.get(ch, 0) + int(c) * w

    if not chain_votes:
        return []

    ranked = sorted(chain_votes.items(), key=lambda kv: (-kv[1], kv[0]))
    return [ch for ch, _ in ranked]


def _safe_chain_int(x):
    try:
        if x is None:
            return None
        if isinstance(x, str):
            x = x.strip()
            if x == "":
                return None
            x = int(float(x))
        else:
            if isinstance(x, (float, np.floating)) and not np.isfinite(x):
                return None
            x = int(x)
        return x
    except Exception:
        return None


global_top5_str = [str(x) for x in global_top5]

preds = []
n_any_found = 0
n_chain_mapped = 0
images = sample_sub["image"].astype(str).tolist()

MAX_CHAINS_TO_USE = 2

# --- SPEED: Hash all test images in one threaded batch (same _ahash_64) to reduce per-image overhead,
# repeated dict lookups, and repeated thread scheduling. This preserves exact hashes and voting logic.
test_hashes = {}
if _hash_ready and test_images_root is not None:
    max_workers = min(16, max(1, (os.cpu_count() or 4)))

    def _hash_test_one(img_name: str):
        h = _test_hash_cache.get(img_name)
        if h is not None or img_name in _test_hash_cache:
            return img_name, h
        test_path = os.path.join(test_images_root, img_name)
        h = _ahash_64(test_path)
        return img_name, h

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for img_name, h in ex.map(_hash_test_one, images, chunksize=256):
            _test_hash_cache[img_name] = h
            test_hashes[img_name] = h
else:
    # keep behavior: no hashes available -> all None
    test_hashes = {img: None for img in images}

for img in images:
    hq = test_hashes.get(img)
    chains = infer_chains_for_test_hash(hq)

    out = []
    seen = set()

    picked = False
    used_chains = 0
    if chains:
        for ch in chains:
            if used_chains >= MAX_CHAINS_TO_USE:
                break
            ch = _safe_chain_int(ch)
            if ch is None:
                continue
            top5_hotels = chain_to_top5_hotels.get(ch)
            if top5_hotels:
                for s in top5_hotels:
                    if s not in seen:
                        out.append(s)
                        seen.add(s)
                    if len(out) == 5:
                        break
                used_chains += 1
                picked = True
                n_chain_mapped += 1
            if len(out) == 5:
                break

        if picked:
            n_any_found += 1

    for s in global_top5_str:
        if s not in seen:
            out.append(s)
            seen.add(s)
        if len(out) == 5:
            break

    if len(out) != 5:
        out = (out + global_top5_str)[:5]
        if len(out) < 5:
            out = (out + [global_top5_str[0]] * 5)[:5]

    preds.append(" ".join(out))

print(
    f"Produced chain-mapped additions for {n_chain_mapped}/{len(sample_sub)} test images."
)
print(
    f"Produced non-empty candidate list for {n_any_found}/{len(sample_sub)} test images."
)

submission = sample_sub.copy()
submission["hotel_id"] = preds

assert list(submission.columns) == ["image", "hotel_id"]
assert len(submission) == len(sample_sub)
assert submission["hotel_id"].astype(str).str.split().map(len).eq(5).all()

out_path = "submission.csv"
submission.to_csv(out_path, index=False)
print("Wrote:", out_path)
print(submission.head())



Global top-5 fallback: 36363 18807 60181 53586 64314
Prepared chain->top5 map for 88 chains (including 0)
Using train_images root: /kaggle/input/hotel-id-2021-fgvc8/train_images
Using test_images root: /kaggle/input/hotel-id-2021-fgvc8/test_images


/usr/local/lib/python3.11/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (108576768 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Built train hash index (hash->chain): hashed 41734 train images (cap=90000, per_chain=1200).
Produced chain-mapped additions for 9749/9756 test images.
Produced non-empty candidate list for 9746/9756 test images.
Wrote: submission.csv
                  image                       hotel_id
0  f1608c9f17fb6920.jpg   64314 58884 8431 46780 50666
1  c6c63939c67931e1.jpg  54894 25191 52431 49184 20481
2  83c214f3e90717ed.jpg   17437 52090 15594 46921 2941
3  a6b5dbe436592c82.jpg    32493 3983 1845 28756 50147
4  dd2f02f4354993cc.jpg  37476 61136 39435 65329 14611


In [3]:
import subprocess, shlex, os

if os.path.exists("submission.csv"):
    print(
        subprocess.check_output(shlex.split("ls -lha submission.csv")).decode("utf-8")
    )
    print(
        subprocess.check_output(shlex.split("head -n 5 submission.csv")).decode("utf-8")
    )
else:
    raise FileNotFoundError("submission.csv was not created.")

-rw-r--r-- 1 root root 478K May 12 08:27 submission.csv

image,hotel_id
f1608c9f17fb6920.jpg,64314 58884 8431 46780 50666
c6c63939c67931e1.jpg,54894 25191 52431 49184 20481
83c214f3e90717ed.jpg,17437 52090 15594 46921 2941
a6b5dbe436592c82.jpg,32493 3983 1845 28756 50147

